# 01 — Plasma OES Preprocessing Pipeline

**LIBS Benchmark dataset** — 12-class, 40002-channel OES spectra (200–1000 nm).

This notebook demonstrates:
1. Loading a small LIBS Benchmark subset
2. Applying the `Preprocessor` pipeline: cosmic-ray removal → ALS baseline → SavGol smoothing → SNV normalisation
3. Visualising raw vs preprocessed spectra
4. Detecting emission peaks with `detect_peaks`

In [1]:
import os
import sys
import warnings
from pathlib import Path

# ------------------------------------------------------------------
# Ensure we run from the project root (works whether notebook is
# launched from project root or from notebooks/ directory).
# ------------------------------------------------------------------
nb_dir = Path.cwd()
if nb_dir.name == "notebooks":
    os.chdir(nb_dir.parent)
sys.path.insert(0, str(Path.cwd()))
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")          # headless backend — safe for nbconvert
import matplotlib.pyplot as plt
import numpy as np

print("Working directory:", Path.cwd())

Working directory: C:\Users\PC\libs-spectral-analysis


## 1 · Load LIBS Benchmark (small subset)

In [2]:
from src.data_loader import load_libs_benchmark

# max_spectra_per_class=2 → ~192 samples total (fast load)
X, y, wl = load_libs_benchmark("data/libs_benchmark/", max_spectra_per_class=2)
print(f"Loaded : {X.shape[0]} spectra × {X.shape[1]} channels")
print(f"Wavelength range : {wl[0]:.1f} – {wl[-1]:.1f} nm")
print(f"Classes : {sorted(set(y.tolist()))}")

Loaded : 192 spectra × 40002 channels
Wavelength range : 200.0 – 1000.0 nm
Classes : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]


## 2 · Select one representative spectrum per class

In [3]:
# Pick the first spectrum from each of the 12 classes
n_classes = len(set(y.tolist()))
class_indices = [int(np.where(y == c)[0][0]) for c in range(n_classes)]
X_vis = X[class_indices]
print(f"Reference spectra : {X_vis.shape} (one per class)")

Reference spectra : (12, 40002) (one per class)


## 3 · Apply the Preprocessor pipeline

In [4]:
from src.preprocessing import Preprocessor

# Full pipeline: cosmic-ray removal → ALS baseline → SavGol → SNV
pp = Preprocessor(
    baseline="als",
    normalize="snv",
    denoise="savgol",
    cosmic_ray=True,
)
pp.fit(X_vis)
X_pp = pp.transform(X_vis)
print(f"Preprocessed: shape={X_pp.shape}, dtype={X_pp.dtype}")
print(f"Intensity range after SNV: [{X_pp.min():.2f}, {X_pp.max():.2f}]")

Preprocessed: shape=(12, 40002), dtype=float32
Intensity range after SNV: [-1.44, 50.35]


## 4 · Detect emission peaks (class 0)

In [5]:
from src.features import detect_peaks

# detect_peaks returns a DataFrame with columns: wavelength_nm, intensity, prominence, fwhm_nm
peaks_df = detect_peaks(X_pp[0], wl, min_prominence=0.1)
peaks_wl = peaks_df["wavelength_nm"].values
peaks_int = peaks_df["intensity"].values
print(f"Detected {len(peaks_wl)} peaks in class-0 spectrum")
if len(peaks_wl) > 0:
    top5 = peaks_df.nlargest(5, "intensity")
    print("Top 5 peaks (nm):", top5["wavelength_nm"].round(1).tolist())

Detected 1828 peaks in class-0 spectrum
Top 5 peaks (nm): [393.4, 616.3, 396.9, 766.6, 589.0]


## 5 · Visualise raw vs preprocessed spectra + peak annotations

In [6]:
colors = plt.cm.tab20(np.linspace(0, 1, n_classes))
class_labels = [f"class_{c:02d}" for c in range(n_classes)]
WL_STEP = 20   # subsample for display speed

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# ---- top-left: raw spectra (all classes) ----
ax = axes[0, 0]
for i in range(n_classes):
    ax.plot(wl[::WL_STEP], X_vis[i, ::WL_STEP],
            color=colors[i], alpha=0.75, lw=0.8, label=class_labels[i])
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Intensity (a.u.)")
ax.set_title("Raw Spectra — 12 classes")
ax.legend(fontsize=6, ncol=2, loc="upper right")
ax.grid(True, alpha=0.3)

# ---- top-right: preprocessed spectra (all classes) ----
ax = axes[0, 1]
for i in range(n_classes):
    ax.plot(wl[::WL_STEP], X_pp[i, ::WL_STEP],
            color=colors[i], alpha=0.75, lw=0.8, label=class_labels[i])
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("SNV Intensity")
ax.set_title("Preprocessed Spectra (ALS + SavGol + SNV)")
ax.legend(fontsize=6, ncol=2, loc="upper right")
ax.grid(True, alpha=0.3)

# ---- bottom-left: raw class-0 ----
ax = axes[1, 0]
ax.plot(wl[::10], X_vis[0, ::10], color="steelblue", lw=0.8, alpha=0.9)
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Intensity (a.u.)")
ax.set_title("Class 0 — Raw Spectrum")
ax.grid(True, alpha=0.3)

# ---- bottom-right: preprocessed class-0 + peak annotations ----
ax = axes[1, 1]
ax.plot(wl, X_pp[0], color="tomato", lw=0.8, alpha=0.9, label="Preprocessed")
if len(peaks_wl) > 0:
    # Draw vertical lines for detected peaks
    for pw in peaks_wl[:30]:
        ax.axvline(pw, color="green", alpha=0.3, lw=0.7)
    # Annotate top-5 peaks
    top5 = np.argsort(peaks_int)[-5:][::-1]
    for idx in top5:
        ax.annotate(
            f"{peaks_wl[idx]:.1f}",
            xy=(peaks_wl[idx], peaks_int[idx]),
            xytext=(peaks_wl[idx] + 8, peaks_int[idx] + 0.06),
            fontsize=7,
            color="darkgreen",
            arrowprops=dict(arrowstyle="->", color="darkgreen", lw=0.7),
        )
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("SNV Intensity")
ax.set_title("Class 0 — Preprocessed + Peak Annotations")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle(
    "Plasma OES Preprocessing Pipeline — LIBS Benchmark",
    fontsize=13, fontweight="bold",
)
plt.tight_layout()

out_path = Path("outputs/notebook_01_preprocessing.png")
out_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(str(out_path), dpi=100, bbox_inches="tight")
plt.close(fig)
print(f"Saved figure → {out_path}")

Saved figure → outputs\notebook_01_preprocessing.png


## Summary

| Step | Method | Key parameter |
|------|--------|---------------|
| Cosmic-ray removal | Z-score median filter | threshold = 5 σ |
| Baseline correction | Asymmetric Least Squares | λ = 1×10⁶, p = 0.01 |
| Smoothing | Savitzky–Golay | window = 11, poly = 3 |
| Normalisation | Standard Normal Variate | — |
| Peak detection | `scipy.signal.find_peaks` | prominence ≥ 0.10 |